# Day 065 · 时点 vs 时段数据 · 中国版
**Snapshot vs Period** · 阶段 P3 · 数据与因子工程

> 进入因子工程,第1 块地基是搞懂数据的两种身份。一种叫时点数据,是某一天的快照,比如股价、市值、市净率,它回答的是「此刻是多少」;另一种叫时段数据,是一段时间的累积,比如收益率、成交额、波动率,它回答的是「这段时间变了多少」。这俩最容易混的就是市值。很多人以为市值是一只票身上一个固定的标签,其实它是每天随股价上下浮动的时点快照,一年下来排名能洗牌好几轮。这一节我们就用八只跨行业的真实股票,把市值这个时点快照、和区间收益这个时段位移对齐起来,亲眼看一个最经典也最隐蔽的坑——前视偏差:你要是用回测结束那天(未来)的市值,去给股票分大盘小盘,等于偷看了未来,选出来的组合和回测结论会整个失真。

---

### 关于「中国版」

本 notebook 是为**国内学员**优化的版本:
- 数据源用 **akshare**(国内可访问、零 VPN、免注册),取代了视频里的 yfinance
- 标的尽量保持原意:美股 ETF→A 股 ETF / 国际公司→A 股龙头
- 所讲的**概念和方法 100% 一致**,但**具体数字可能与视频里略有差异**(因为是不同时间窗 / 不同标的)
- 一般情况国内 `pip install akshare` 即可,无需 token / VPN

**课件生成日期:** 2026-06-18  ·  **建议学习时长:** 18 分钟

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有必需的 Python 包(含 `akshare`),缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续


In [1]:
# === 环境自检 + 自动安装(运行此单元格即可)===
import importlib, subprocess, sys, os

REQUIRED = ["akshare", "baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels"]
PIP_NAME = {"sklearn":"scikit-learn","cv2":"opencv-python","PIL":"Pillow","bs4":"beautifulsoup4","yaml":"PyYAML"}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))
if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置 ===
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
CJK = ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
       "C:/Windows/Fonts/msyh.ttc","C:/Windows/Fonts/simhei.ttf",
       "/System/Library/Fonts/PingFang.ttc","/System/Library/Fonts/STHeiti Medium.ttc"]
for p in CJK:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP","Microsoft YaHei","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪")


✓ 所有 9 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪


## 🔌 第二步:加载国内数据助手

下面这一格是**工具函数**(可以折叠,不需要修改)。它把 `yfinance` 风格的 ticker(如 `600519.SS`)自动路由到对应的 akshare 接口,提供 `get_close(ticker)` 和 `get_close_multi(tickers_dict)` 两个函数。

In [2]:
# === 国内数据源助手(akshare 后端,不需要 VPN)===
# 这一格是工具函数,可以折叠,不需要修改。
# 它把 yfinance 风格的 ticker(如 "600519.SS" / "0700.HK" / "AAPL" / "BTC-USD")
# 自动路由到对应的 akshare 接口,统一返回 yfinance 风格的 Close DataFrame。

import re
from datetime import datetime, timedelta
import pandas as pd
import akshare as ak

_TICKER_MAP = {
    "^GSPC": ("us_index_sina", ".INX"),
    "^DJI":  ("us_index_sina", ".DJI"),
    "^IXIC": ("us_index_sina", ".IXIC"),
    "GC=F":  ("foreign_futures", "GC"),
    "SI=F":  ("foreign_futures", "SI"),
    "CL=F":  ("foreign_futures", "CL"),
    "BTC-USD": ("crypto", "BTC"),
    "ETH-USD": ("crypto", "ETH"),
}

def _retry(fn, *args, _retries=4, _wait=1.5, **kwargs):
    """akshare 上游(东方财富/新浪/Binance)偶有 RemoteDisconnected / Timeout,自动重试 4 次。
    2026-05-11 加:用户跑 cn notebook 拉 002594.SZ 时上游断连 → 整节卡死。
    每次重试间隔 _wait 秒(指数退避 1x → 1.5x → 2.25x)。
    """
    import time as _t
    last_err = None
    wait = _wait
    for i in range(_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            name = type(e).__name__
            if i == _retries - 1:
                print(f"  ✗ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次仍失败({name}),放弃")
                raise
            print(f"  ⚠ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次失败({name}),{wait:.1f}s 后重试")
            _t.sleep(wait)
            wait *= 1.5

def _parse_period(period):
    end = datetime.today()
    m = re.match(r"^(\d+)\s*(y|mo|d|w)$", period.lower().strip())
    days = 365 * 3 if not m else int(m.group(1)) * {"y":365,"mo":30,"w":7,"d":1}[m.group(2)]
    return (end - timedelta(days=days+30)).strftime("%Y%m%d"), end.strftime("%Y%m%d")

def _classify(ticker):
    t = ticker.strip()
    if t in _TICKER_MAP: return _TICKER_MAP[t]
    if t.endswith((".SS",".SH",".SZ")):
        code = t.split(".")[0]
        if code.startswith(("51","159","58")) or code in ("510300","510500","510050","511010","513100"):
            return ("a_etf", code)
        if code in ("000300","000016","000905","000852","000001"):
            return ("a_index", code)
        return ("a_stock", code)
    if t.endswith(".HK"):
        return ("hk", t.split(".")[0].zfill(5))
    return ("us", t)

def _norm(df, dc, cc):
    out = df[[dc, cc]].copy()
    out[dc] = pd.to_datetime(out[dc])
    return out.set_index(dc).sort_index()[cc].astype(float).rename("Close")

def get_close(ticker, period="3y"):
    """返回某标的 Close 价格 series。后端 akshare,中国可访问。
    所有 ak.* 调用都过 _retry(4 次,指数退避)— 防东方财富/新浪上游瞬时断连。
    """
    start, end = _parse_period(period)
    kind, sym = _classify(ticker)
    if kind == "a_stock":
        return _norm(_retry(ak.stock_zh_a_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_etf":
        return _norm(_retry(ak.fund_etf_hist_em, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_index":
        idx_map = {"000300":"sh000300","000016":"sh000016","000905":"sh000905","000852":"sh000852","000001":"sh000001"}
        s = _norm(_retry(ak.stock_zh_index_daily_em, symbol=idx_map.get(sym, f"sh{sym}")), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "hk":
        return _norm(_retry(ak.stock_hk_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "us":
        s = _norm(_retry(ak.stock_us_daily, symbol=sym, adjust="qfq"), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "us_index_sina":
        s = _norm(_retry(ak.index_us_stock_sina, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "foreign_futures":
        s = _norm(_retry(ak.futures_foreign_hist, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "crypto":
        import requests as _rq
        def _binance():
            r = _rq.get("https://api.binance.com/api/v3/klines",
                        params={"symbol": f"{sym}USDT", "interval": "1d", "limit": 1000}, timeout=15)
            r.raise_for_status()
            return r.json()
        klines = _retry(_binance)
        df = pd.DataFrame(klines, columns=["open_time","open","high","low","close","volume",
                                            "close_time","qav","trades","tbb","tbq","ignore"])
        df["date"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close"] = df["close"].astype(float)
        s = df.set_index("date").sort_index()["close"].rename("Close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    raise ValueError(f"unsupported ticker: {ticker}")

def get_close_multi(tickers, period="3y"):
    """批量取 Close,返回 DataFrame,列名是 tickers dict 的 key(中文名),按交集日期对齐。"""
    series = {name: get_close(t, period=period) for name, t in tickers.items()}
    return pd.concat(series, axis=1).sort_index()

print("✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据")


✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据


## 学习目标

- 分清两类数据:时点是快照(市值/股价/市净率,问此刻多少),时段是区间(收益率/成交额/波动,问这段变多少),两者对齐方式完全不同
- 明白市值不是固定标签,而是每天随股价浮动的时点快照,一年里大小盘排名一直在漂
- 理解因子对齐的正确姿势:在调仓那天用当天已知的因子值打标签,再看持有期这一段时间的收益
- 看清前视偏差:用回测期末(未来)的市值给股票分大小盘,等于偷看未来,会让分组收益严重失真
- 建立铁律:任何因子分组、打标签,只能用调仓时点当时已经知道的数据,绝不能用未来的快照

## 历史背景:小赵图省事用最新市值回测小盘策略,把后来涨成大盘的牛股提前踢出小盘组,回测结论全错

小赵想验证一个想法:小盘股是不是比大盘股涨得好。他从数据源拉了一份股票列表,顺手用了上面现成的、也就是最新的市值,挑出市值最小的一批当小盘组,回测过去几年的表现。结果回测显示小盘组表现平平,他有点失望,差点就把小盘这个方向否了。后来一位做因子的前辈看了他的代码,问了一句:你回测的是过去几年,可你分组用的是哪天的市值?小赵说,用的最新的。前辈摇头:这就坏了。你想想,有的票几年前还是个小不点,后来涨成了大盘股,你用最新市值一看它是大盘,就把它从小盘组里踢出去了——可它当年明明是小盘,而且正是它涨得最猛。你这等于偷看了未来,提前把后来的大牛股从小盘组开除了,小盘组当然显得不行。小赵这才恍然大悟,原来不是小盘策略不行,是他用未来的市值给过去贴了标签。这节课,我们就把小赵踩的这个前视偏差,用八只真实股票一步步还原清楚。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 1. 数据的两种身份:时点(快照)vs 时段(区间)

做因子之前,先把数据分清两类。时点数据是某一天的一张快照,比如某天的收盘价、某天的市值、某天的市净率,它只回答此刻是多少,明天又是另一张快照。时段数据是一段时间的累积量,比如这个月的收益率、这一年的成交额、这一季的波动率,它回答的是这段时间一共变了多少,必须有起点和终点两个时间才算得出来。这俩的对齐方式完全不同:时点数据你要指定是哪一天的,时段数据你要指定是哪一段的。混了,就会算出莫名其妙的结果。


### 2. 2. 市值是时点快照,不是固定标签

最容易被当成固定标签的就是市值。很多人脑子里,某只票就是大盘股、某只票就是小盘股,好像刻在身上一样。其实市值等于股价乘以总股本,股价每天在动,市值就每天在动。一只票今天是小盘,涨一年可能就成了中盘甚至大盘;一只大白马跌两年也可能缩回中小盘。所以市值是一张每天刷新的快照,大小盘的排名一直在漂,你用哪一天的市值给它贴标签,得到的分组可能完全不同。


### 3. 3. 因子对齐:调仓日打标签,持有期看收益

造任何因子做回测,本质都是一件事:在调仓那一天,用当天已经知道的因子值(时点)给股票排序分组,然后看接下来一段时间(时段)各组的收益。注意两个时间点:打标签用的是调仓日当天的快照,看收益用的是调仓日往后那一段。打标签的数据必须是调仓那一刻真实能拿到的,绝不能用之后才知道的。把这两件事的时间对齐对,因子回测才算干净。


### 4. 4. 前视偏差:用未来的快照分组就是偷看未来

前视偏差,说白了就是你在回测里用了当时根本拿不到、要等到未来才知道的信息。市值这里最典型:你回测过去几年的小盘策略,却图省事用了今天(也就是回测期末、未来)的市值来分大小盘。这等于你站在终点回头给起点的股票贴标签,自然带着上帝视角。一只后来涨成大盘的牛股,用未来市值一看是大盘,就被你提前从小盘组开除了;一只后来跌成小盘的熊股,反而被塞进了小盘组。结果就是回测和现实严重脱节。


### 5. 5. 跨档的票是前视偏差的重灾区

为什么前视偏差有时影响巨大?关键在那些市值跨了档的票——几年里从小盘长成大盘、或者从大盘缩成小盘。对没跨档的票,用期初还是期末市值分组都进同一组,影响不大;可一旦有牛股从小盘成长跨进大盘、或熊股从大盘跌进小盘,用未来市值分组就会把它们换错组,而恰恰是这些涨跌最猛的票,对组合收益影响最大。所以越是行情分化、个股涨跌悬殊的几年,前视偏差越致命。


## 实操:市值是时点快照,收益是时段位移:用错日期的市值给股票分组就是前视偏差(期初 vs 期末市值分组,看回测收益差多少)

本节无外部数据,纯模拟/统计运算,国内国外都能跑。**直接 Run All** 看结果。

**依赖:** `pip install pandas numpy matplotlib akshare statsmodels scipy`

In [3]:
# day_065_snapshot_vs_period.py — 时点 vs 时段数据:市值快照陷阱(前视偏差)
# 真名上屏:baostock / query_history_k_data_plus / adjustflag(3不复权/2前复权) / query_profit_data → totalShare / rank 排名 / nsmallest 分组
# 核心类比:市值是"此刻的快照"(时点),收益是"一段路的位移"(时段);用错日期的市值给股票贴大小盘标签,等于偷看未来 = 前视偏差
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs


def _data_path(_name):
    # 铁律62:data/ 放在 notebook 文件夹里。仓库根(run_lab)存取 out/notebook/data/;
    # 原版 notebook 在 out/notebook/ 用 data/;中国版在 out/notebook/cn/ 用 ../data/
    from pathlib import Path as _P
    _here = _P.cwd()
    for _b in [_here/'data', _here/'..'/'data', _here/'out'/'notebook'/'data', _here/'..'/'..'/'data', _here/'..'/'..'/'..'/'data']:
        if (_b/_name).exists():
            return str(_b/_name)
    if (_here/'out'/'notebook').exists():
        _t = _here/'out'/'notebook'/'data'
    elif _here.name == 'cn':
        _t = _here/'..'/'data'
    else:
        _t = _here/'data'
    _t.mkdir(parents=True, exist_ok=True)
    return str(_t/_name)


pd.set_option('display.width', 160)
plt.rcParams['axes.unicode_minus'] = False

# ==== 标的池:8 只跨行业散户熟悉的票(市值四年洗牌明显)====
STOCKS = {
    '长城汽车': 'sh.601633', '药明康德': 'sh.603259', '福耀玻璃': 'sh.600660',
    '通威股份': 'sh.600438', '中国神华': 'sh.601088', '顺丰控股': 'sz.002352',
    '北方华创': 'sz.002371', '用友网络': 'sh.600588',
}
START, END = '2021-01-01', '2024-12-31'
CSV_MCAP = _data_path('D065_market_cap.csv')   # 日市值(亿元)= 不复权收盘 × 总股本
CSV_QFQ = _data_path('D065_qfq_close.csv')     # 前复权收盘(算真实区间收益)


# ==== 0. 拉数据:不复权收盘算市值,前复权收盘算收益,季度总股本 ffill 到日 ====
def _fetch_one(code):
    def k(adj, col):
        rs = bs.query_history_k_data_plus(code, 'date,' + col, start_date=START, end_date=END, frequency='d', adjustflag=adj)
        r = []
        while rs.error_code == '0' and rs.next():
            r.append(rs.get_row_data())
        s = pd.DataFrame(r, columns=['date', col])
        s[col] = pd.to_numeric(s[col], errors='coerce')
        return s.set_index('date')[col]
    raw = k('3', 'close')          # 不复权:真实市值用
    qfq = k('2', 'close')          # 前复权:真实收益用
    shares = {}
    for y in range(2020, 2025):
        for q in range(1, 5):
            rp = bs.query_profit_data(code=code, year=y, quarter=q)
            while rp.error_code == '0' and rp.next():
                row = rp.get_row_data()
                shares[row[2]] = float(row[9])   # statDate -> totalShare 总股本
    ss = pd.Series(shares).sort_index()
    ss.index = pd.to_datetime(ss.index)
    idx = pd.to_datetime(raw.index)
    ts = ss.reindex(idx.union(ss.index)).ffill().reindex(idx)
    ts.index = raw.index
    mcap = raw * ts.values / 1e8     # 亿元
    return mcap, qfq


if os.path.exists(CSV_MCAP) and os.path.exists(CSV_QFQ):
    print('[数据] 从本地 CSV 读取市值/收益')
    mcap = pd.read_csv(CSV_MCAP, parse_dates=['date']).set_index('date')
    qfq = pd.read_csv(CSV_QFQ, parse_dates=['date']).set_index('date')
else:
    print('[数据] baostock 拉取 8 只票 不复权/前复权收盘 + 总股本 ...')
    bs.login()
    mc, qf = {}, {}
    for name, code in STOCKS.items():
        mc[name], qf[name] = _fetch_one(code)
    bs.logout()
    mcap = pd.DataFrame(mc)
    qfq = pd.DataFrame(qf)
    mcap.index.name = 'date'
    qfq.index.name = 'date'
    mcap.to_csv(CSV_MCAP)             # 拉完保存 CSV(铁律62)
    qfq.to_csv(CSV_QFQ)
    print('[数据] 已存 CSV ->', CSV_MCAP)

mcap.index = pd.to_datetime(mcap.index)    # 两条路径(CSV/fetch)统一成日期索引
qfq.index = pd.to_datetime(qfq.index)
mcap = mcap.dropna()
qfq = qfq.reindex(mcap.index)
T0, T1 = mcap.index[0], mcap.index[-1]     # 期初 / 期末(用真实交易日)

print('\n==== 市值快照陷阱:时点(市值)对齐时段(收益)的前视偏差 ====')
print('标的池 %d 只 · %s ~ %s' % (len(STOCKS), T0.date(), T1.date()))

# ==== 1. 市值是时点快照:同一只票,期初 vs 期末两个快照天差地别 ====
cap0 = mcap.loc[T0]
cap1 = mcap.loc[T1]
ret = qfq.loc[T1] / qfq.loc[T0] - 1.0       # 真实区间收益(前复权)
tbl = pd.DataFrame({'T0市值亿': cap0, 'T1市值亿': cap1, '区间收益%': ret * 100})
tbl['T0名次'] = tbl['T0市值亿'].rank(ascending=False).astype(int)   # 1=最大
tbl['T1名次'] = tbl['T1市值亿'].rank(ascending=False).astype(int)
tbl['名次漂移'] = tbl['T0名次'] - tbl['T1名次']                     # 正=变大, 负=变小
tbl = tbl.sort_values('T0名次')
print('\n[每只票 时点快照 + 时段收益]')
print(tbl.round(1).to_string())

# ==== 2. 前视偏差:用期初市值(正确)vs 期末市值(偷看未来)给小盘分组 ====
half = len(STOCKS) // 2
small_T0 = set(tbl.nsmallest(half, 'T0市值亿').index)   # 正确:回测开始那天就知道的市值
small_T1 = set(tbl.nsmallest(half, 'T1市值亿').index)   # 前视:用了回测结束才知道的市值
big_T0 = set(STOCKS) - small_T0
r_small_T0 = float(np.mean([ret[n] for n in small_T0]) * 100)
r_big_T0 = float(np.mean([ret[n] for n in big_T0]) * 100)
r_small_T1 = float(np.mean([ret[n] for n in small_T1]) * 100)
bias = r_small_T1 - r_small_T0
swap = sorted(small_T0 ^ small_T1)
print('\n[正确·用 T0 期初市值分组] 小盘组区间收益 %.1f%% / 大盘组 %.1f%%' % (r_small_T0, r_big_T0))
print('[前视·用 T1 期末市值分组] 小盘组区间收益 %.1f%%  <- 偷看了未来才知道的市值' % r_small_T1)
print('前视偏差 = %+.1f 个百分点(同一批票、同一段时间,只因分组用错了日期)' % bias)
print('被未来市值换错组的票:', swap)
grower = tbl['名次漂移'].idxmax()
loser_drop = (qfq.loc[T1] / qfq.loc[T0] - 1).idxmin()
print('成长跨档(最该留在小盘却被前视踢出):%s 名次 %d→%d,区间涨 %.0f%%' % (
    grower, tbl.loc[grower, 'T0名次'], tbl.loc[grower, 'T1名次'], tbl.loc[grower, '区间收益%']))

# ==== 3. 三张图 ====
# 图1:市值曲线(时点快照,每天在变)
fig, ax = plt.subplots(figsize=(11, 6))
for n in STOCKS:
    ax.plot(mcap.index, mcap[n], lw=1.3, label=n)
ax.axvline(T0, ls='--', c='green', alpha=.6)
ax.axvline(T1, ls='--', c='red', alpha=.6)
ax.set_title('市值是时点快照:每天随股价在变,期初(绿)和期末(红)两个快照天差地别')
ax.set_ylabel('总市值(亿元)')
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_01.png', dpi=110)
plt.close()

# 图2:前视分组 vs 正确分组 的回测收益
fig, ax = plt.subplots(figsize=(9, 6))
labels = ['正确\n小盘(期初T0)', '正确\n大盘(期初T0)', '前视\n小盘(期末T1)']
vals = [r_small_T0, r_big_T0, r_small_T1]
colors = ['#2e7d32', '#1565c0', '#c62828']
ax.bar(labels, vals, color=colors)
for i, v in enumerate(vals):
    ax.text(i, v + (2 if v >= 0 else -3), '%.1f%%' % v, ha='center', fontweight='bold')
ax.axhline(0, c='k', lw=.8)
ax.set_title('同一批票同一段时间:用期初市值(正确)vs 期末市值(前视偷看未来)分组,收益差 %.0f 个点' % bias)
ax.set_ylabel('区间平均收益(%)')
ax.grid(alpha=.3, axis='y')
fig.tight_layout()
fig.savefig('chart_02.png', dpi=110)
plt.close()

# 图3:市值排名漂移(贴标签用哪天,结果完全不同)
fig, ax = plt.subplots(figsize=(11, 6))
rankts = mcap.rank(axis=1, ascending=False)
for n in STOCKS:
    ax.plot(rankts.index, rankts[n], lw=1.3, label=n)
ax.invert_yaxis()
ax.set_yticks(range(1, len(STOCKS) + 1))
ax.set_title('市值排名四年一直在漂:你用哪天的市值给票贴大/小盘标签,分组结果完全不同')
ax.set_ylabel('市值排名(1=最大)')
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_03.png', dpi=110)
plt.close()

print('\n[done] 市值快照(时点)对齐区间收益(时段),前视偏差实证完成 —— 3 图已出')

[数据] 从本地 CSV 读取市值/收益

==== 市值快照陷阱:时点(市值)对齐时段(收益)的前视偏差 ====
标的池 8 只 · 2021-01-04 ~ 2024-12-31

[每只票 时点快照 + 时段收益]
       T0市值亿   T1市值亿  区间收益%  T0名次  T1名次  名次漂移
顺丰控股  4122.7  2009.4  -52.7     1     4    -3
中国神华  3578.1  8638.8  229.8     2     1     1
长城汽车  3497.0  2252.8  -28.1     3     2     1
药明康德  3347.8  1589.6  -49.6     4     6    -2
通威股份  1874.9   995.4  -38.4     5     7    -2
用友网络  1494.9   366.6  -76.2     6     8    -2
福耀玻璃  1278.1  1628.5   35.8     7     5     2
北方华创   956.9  2086.4  103.9     8     3     5

[正确·用 T0 期初市值分组] 小盘组区间收益 6.3% / 大盘组 24.9%
[前视·用 T1 期末市值分组] 小盘组区间收益 -32.1%  <- 偷看了未来才知道的市值
前视偏差 = -38.4 个百分点(同一批票、同一段时间,只因分组用错了日期)
被未来市值换错组的票: ['北方华创', '药明康德']
成长跨档(最该留在小盘却被前视踢出):北方华创 名次 8→3,区间涨 104%

[done] 市值快照(时点)对齐区间收益(时段),前视偏差实证完成 —— 3 图已出


## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 前视偏差·小盘组 | N/A | 前视偏差实证:8只票2021→2024,用期初市值分小盘组+6.3%,用期末市值(偷看未来)分小盘组-32.1%,同一批票同段时间凭空差出约38个百分点。 |
| 成长跨档·北方华创 | sz.002371 | 北方华创2021市值最小957亿(第8名),4年涨104%升到2086亿(第3名),用期末市值分组会把这个大牛踢出小盘组,正是前视偏差的破坏。 |
| 衰退跨档·药明康德 | sh.603259 | 药明康德从3348亿大盘4年腰斩到1590亿小盘(-50%),用期末市值分组反被塞进小盘组拖后腿,和北方华创一出一进做坏小盘组。 |
| 快照剧变·中国神华 | sh.601088 | 中国神华赶上煤炭行情4年市值翻2.4倍(3578→8639亿,+229.8%,名次2→1),用友网络缩到原来1/4(1495→367亿,-76.2%),市值是每天在变的时点快照。 |


## 常见坑

### ⚠ 01. 用最新市值回测历史选股,提前踢出后来才长大的牛股

分组打标签必须用调仓当天的市值,用今天的(未来的)市值给过去贴标签,就是经典前视偏差。

### ⚠ 02. 把市值当固定标签,忽略它每天在变、排名一直漂

市值=股价×总股本,股价天天动市值就天天动,大小盘是会漂移的快照不是刻死的身份。

### ⚠ 03. 时点因子和时段收益对齐错位

打标签用调仓日的时点快照,看收益用持有期这一时段,两个时间一旦对错就全乱。

### ⚠ 04. 调仓日用了收盘后才知道的数据,隐性偷看未来

哪怕日期对,若用了当天收盘价才能算出、但开盘时还没有的值,也是一种隐性未来函数。

### ⚠ 05. 只看一个时点的市值排名就下结论

单看某一天的市值排名容易以偏概全,要意识到换一天排名可能就变了,结论别太武断。

## 实战 SOP · 时点 vs 时段 — 几条铁规矩

1. 先分清手里的数据是时点(快照)还是时段(区间):市值/价/市净率是时点,收益/成交额/波动是时段。
2. 因子分组打标签,只能用调仓那一天当时已知的时点值,绝不能用未来某天的快照。
3. 对齐要对上:打标签用调仓日时点,算收益用持有期时段,两个时间分清楚。
4. 特别警惕跨档的票(小变大、大变小),它们是前视偏差影响最大的地方。
5. 回测结论出来前,先问一句:我分组用的市值,是回测当时真能拿到的吗?

> 把这段打印贴在你电脑边。

## 总结 · 你应该带走的

2. 数据分两种:时点是某天快照(市值/价/市净率),时段是一段区间(收益/成交/波动),对齐方式完全不同。
3. 市值=股价×总股本,每天都在变,大小盘是会漂的快照不是刻死的标签;8只票4年排名大洗牌。
4. 因子对齐:换仓日用当天已知值打标签(时点),持有期看各组收益(时段),两个时间分清楚。
5. 前视偏差=用回测期末(未来)市值给股票分组,等于偷看未来;小盘组收益从+6.3%失真成-32.1%。
6. 跨档的票(小变大、大变小)是前视偏差重灾区:北方华创涨104%被踢出小盘、药明康德腰斩被塞进。
7. 铁律:任何因子分组打标签,只能用换仓那一刻真能拿到的数据,绝不用未来快照。

## 自测题

**Q1.** {'q': '市值是时点数据还是时段数据?为什么?', 'a': '时点数据。市值=股价×总股本,股价每天变市值就每天变,它是某一天的一张快照,回答此刻是多少;而收益率才是时段数据,回答一段时间变了多少。'}

**Q2.** {'q': '为什么用回测期末的市值给股票分大小盘会出错?', 'a': '因为期末市值是回测结束才知道、当时拿不到的未来信息,用它分组就是前视偏差:后来涨成大盘的牛股会被提前踢出小盘组、后来跌成小盘的熊股被塞进来,分组收益严重失真。本课实测小盘组从+6.3%变成-32.1%,差38个点。'}

**Q3.** {'q': '正确的因子分组应该怎么做?', 'a': '每到一个换仓日,就用那一天当时已经能拿到的市值,重新给股票排序分组,标签跟着时间一格格往前滚,绝不用任何要等到未来才知道的数据。'}

把答案写下来,3 天后再回看。

## 下一节预告

**Day 066 · point-in-time 防未来函数** (PIT Data)

这一节我们用市值看清了时点和时段的区别,也踩平了前视偏差这个坑。下一节我们把只用当时已知数据这件事,正式做成一套规范,叫 point-in-time,也就是时点数据规范。我们会对比同一个策略,用了未来函数的回测和严格只用当时数据的回测,收益能差出多少,让你彻底养成不偷看未来的习惯。

## 推荐阅读

- Marcos López de Prado《Advances in Financial Machine Learning》(2018/Wiley)— 专门讲回测里的各种偏差和未来函数陷阱,前视偏差是其中重点,实战派必读。
- Campbell Harvey, Yan Liu《Backtesting》(2015/SSRN)— 系统讨论回测可信度,前视偏差和数据窥探如何制造虚假的好结果。
- Andrew Ang《Asset Management: A Systematic Approach to Factor Investing》(2014/Oxford)— 因子投资经典,讲清因子构建里时点对齐的重要性。
- Eugene Fama, Kenneth French《The Cross-Section of Expected Stock Returns》(1992/JoF)— 规模因子的开山之作,理解市值因子从哪来。
- Tushare / baostock 官方文档 — 国内行情与财务数据接口,搞清每个字段是时点还是时段、对应哪个日期,是动手做因子的第一步。